In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

n_users, n_days = 150, 90
feats = ["logons", "offhours_logons", "usb_events", "files_copied", "email_attachments", "distinct_pcs"]

# each user gets personal baseline rates
base = pd.DataFrame({
    "logons": rng.uniform(2, 6, n_users),
    "offhours_logons": rng.uniform(0.0, 0.5, n_users),
    "usb_events": rng.uniform(0.0, 1.0, n_users),
    "files_copied": rng.lognormal(2.0, 0.8, n_users),     # a few users copy a lot, on purpose
    "email_attachments": rng.uniform(0.5, 4, n_users),
    "distinct_pcs": rng.uniform(1.0, 2.0, n_users),
})

rows = []
insiders = rng.choice(n_users, 8, replace=False)
start_bad = rng.integers(65, 80, size=8)                  # each insider turns bad at a different day
bad_start = dict(zip(insiders, start_bad))

for u in range(n_users):
    for d in range(n_days):
        r = {f: rng.poisson(base.loc[u, f]) for f in feats}
        r.update(user=u, day=d, label=0)
        if u in bad_start and d >= bad_start[u]:           # insider behavior: after-hours, USB, mass copying
            r["offhours_logons"] += rng.poisson(3)
            r["usb_events"] += rng.poisson(4)
            r["files_copied"] += rng.poisson(base.loc[u, "files_copied"] * 3 + 20)
            r["distinct_pcs"] += rng.poisson(2)
            r["label"] = 1
        rows.append(r)

df = pd.DataFrame(rows).sort_values(["user", "day"]).reset_index(drop=True)
print(df.shape, "| malicious user-days:", df["label"].sum(), "| insiders:", len(insiders))

(13500, 9) | malicious user-days: 120 | insiders: 8


In [2]:
g = df.groupby("user")[feats]
mean = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).mean())
std = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).std())
z = (df[feats] - mean) / std.clip(lower=1.0)              # clip avoids dividing by tiny std
z = z.add_suffix("_z").fillna(0)
data = pd.concat([df, z], axis=1)
z_cols = list(z.columns)
print(data[data["label"] == 1][z_cols].mean().round(1))    # attack days should show large z
print(data[data["label"] == 0][z_cols].mean().round(1))    # normal days should sit near 0

logons_z               0.1
offhours_logons_z      1.7
usb_events_z           1.8
files_copied_z         3.0
email_attachments_z   -0.0
distinct_pcs_z         1.0
dtype: float64
logons_z               0.0
offhours_logons_z     -0.0
usb_events_z           0.0
files_copied_z         0.0
email_attachments_z    0.0
distinct_pcs_z         0.0
dtype: float64


In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

train = data[data["day"] < 60]
test = data[data["day"] >= 60].copy()

def run(cols, name):
    iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1).fit(train[cols])
    test[name] = -iso.score_samples(test[cols])            # higher = more abnormal
    print(f"{name}: PR-AUC = {average_precision_score(test['label'], test[name]):.4f}")

run(feats, "score_raw")
run(z_cols, "score_peruser")
print("baseline (random):", round(test["label"].mean(), 4))

score_raw: PR-AUC = 0.9829
score_peruser: PR-AUC = 0.5768
baseline (random): 0.0267


In [4]:
def recall_at_k(col, frac):
    k = int(len(test) * frac)
    top = test.nlargest(k, col)
    return top["label"].sum() / test["label"].sum()

for col in ["score_raw", "score_peruser"]:
    print(col, {f"top {int(f*100)}%": round(recall_at_k(col, f), 3) for f in [0.01, 0.05]})

# did we catch every insider at least once?
top5 = test.nlargest(int(len(test) * 0.05), "score_peruser")
print("insiders flagged:", top5.loc[top5["label"] == 1, "user"].nunique(), "of", len(insiders))

score_raw {'top 1%': np.float64(0.375), 'top 5%': np.float64(0.983)}
score_peruser {'top 1%': np.float64(0.333), 'top 5%': np.float64(0.658)}
insiders flagged: 8 of 8


In [5]:
import os
os.makedirs("../data/processed", exist_ok=True)
test[["user", "day", "label", "score_peruser"]].to_csv("../data/processed/insider_scores.csv", index=False)

In [6]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

# ---------- PART 1: data generator, now with an "intensity" knob ----------
# intensity=1.0 is what you ran before. Lower values make the insider's behavior subtler.
def make_data(intensity=1.0, seed=42):
    rng = np.random.default_rng(seed)               # fixed seed => repeatable results

    # each user gets their own personal "normal" rates
    base = pd.DataFrame({
        "logons": rng.uniform(2, 6, n_users),
        "offhours_logons": rng.uniform(0.0, 0.5, n_users),
        "usb_events": rng.uniform(0.0, 1.0, n_users),
        "files_copied": rng.lognormal(2.0, 0.8, n_users),
        "email_attachments": rng.uniform(0.5, 4, n_users),
        "distinct_pcs": rng.uniform(1.0, 2.0, n_users),
    })

    insiders = rng.choice(n_users, 8, replace=False)             # pick 8 users to turn malicious
    bad_start = dict(zip(insiders, rng.integers(65, 80, size=8)))  # day each one turns bad

    rows = []
    for u in range(n_users):
        for d in range(n_days):
            # a normal day: each count is a random draw around that user's own rate
            r = {f: rng.poisson(base.loc[u, f]) for f in feats}
            r.update(user=u, day=d, label=0)
            # after the start day, an insider adds extra suspicious activity
            if u in bad_start and d >= bad_start[u]:
                r["offhours_logons"] += rng.poisson(3 * intensity)
                r["usb_events"] += rng.poisson(4 * intensity)
                r["files_copied"] += rng.poisson((base.loc[u, "files_copied"] * 3 + 20) * intensity)
                r["distinct_pcs"] += rng.poisson(2 * intensity)
                r["label"] = 1                                    # ground truth, used ONLY for scoring
            rows.append(r)
    return pd.DataFrame(rows).sort_values(["user", "day"]).reset_index(drop=True)


# ---------- PART 2: build features, train, and compare ----------
def compare(df):
    # (a) ROLLING baseline: compare today to this user's previous 30 days.
    #     shift(1) = use only PAST days, so today never leaks into its own baseline.
    g = df.groupby("user")[feats]
    mean = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).mean())
    std = g.transform(lambda s: s.shift(1).rolling(30, min_periods=10).std())
    roll = ((df[feats] - mean) / std.clip(lower=1.0)).add_suffix("_roll").fillna(0)

    # (b) FROZEN baseline: learn each user's normal ONLY from the clean first 60 days,
    #     then never update it. Attack days cannot pollute it. No leakage either,
    #     because test days (60+) are never used to compute it.
    clean = df[df["day"] < 60]
    user_mean = clean.groupby("user")[feats].mean()
    user_std = clean.groupby("user")[feats].std().clip(lower=1.0)   # clip avoids dividing by ~0
    frozen = (df[feats].values - user_mean.loc[df["user"]].values) / user_std.loc[df["user"]].values
    frozen = pd.DataFrame(frozen, columns=[f + "_frozen" for f in feats], index=df.index)

    data = pd.concat([df, roll, frozen], axis=1)

    # the four feature sets we compare
    sets = {
        "raw": feats,
        "rolling_z": list(roll.columns),
        "frozen_z": list(frozen.columns),
        "raw+frozen": feats + list(frozen.columns),    # hybrid: absolute size AND personal deviation
    }

    # train on clean early days only (no labels used), test on days 60+
    train = data[data["day"] < 60]
    test = data[data["day"] >= 60].copy()
    k = int(len(test) * 0.05)                          # "top 5%" = the alerts an analyst can read

    for name, cols in sets.items():
        iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1).fit(train[cols])
        test["s"] = -iso.score_samples(test[cols])     # higher = more abnormal
        pr = average_precision_score(test["label"], test["s"])
        top = test.nlargest(k, "s")
        rec = top["label"].sum() / test["label"].sum() # share of malicious days inside the top 5%
        caught = top.loc[top["label"] == 1, "user"].nunique()   # distinct insiders flagged
        print(f"  {name:12s} PR-AUC={pr:.3f}  top5% recall={rec:.3f}  insiders caught={caught}/8")


# ---------- PART 3: run at three attack strengths ----------
for intensity in [1.0, 0.5, 0.25]:
    print(f"\nattack intensity = {intensity}")
    compare(make_data(intensity))


attack intensity = 1.0
  raw          PR-AUC=0.983  top5% recall=0.983  insiders caught=8/8
  rolling_z    PR-AUC=0.577  top5% recall=0.658  insiders caught=8/8
  frozen_z     PR-AUC=0.985  top5% recall=1.000  insiders caught=8/8
  raw+frozen   PR-AUC=0.986  top5% recall=1.000  insiders caught=8/8

attack intensity = 0.5
  raw          PR-AUC=0.735  top5% recall=0.783  insiders caught=8/8
  rolling_z    PR-AUC=0.426  top5% recall=0.525  insiders caught=8/8
  frozen_z     PR-AUC=0.819  top5% recall=0.892  insiders caught=8/8
  raw+frozen   PR-AUC=0.797  top5% recall=0.858  insiders caught=8/8

attack intensity = 0.25
  raw          PR-AUC=0.284  top5% recall=0.458  insiders caught=8/8
  rolling_z    PR-AUC=0.200  top5% recall=0.350  insiders caught=8/8
  frozen_z     PR-AUC=0.471  top5% recall=0.633  insiders caught=8/8
  raw+frozen   PR-AUC=0.386  top5% recall=0.542  insiders caught=8/8


In [7]:
def insider_frozen_scores(intensity=0.5, seed=42):
    d = make_data(intensity, seed)
    clean = d[d["day"] < 60]                                   # frozen baseline from clean days only
    mu = clean.groupby("user")[feats].mean()
    sd = clean.groupby("user")[feats].std().clip(lower=1.0)
    z = (d[feats].values - mu.loc[d["user"]].values) / sd.loc[d["user"]].values
    z = pd.DataFrame(z, columns=[f + "_frozen" for f in feats], index=d.index)
    data = pd.concat([d, z], axis=1)
    tr, te = data[data["day"] < 60], data[data["day"] >= 60].copy()
    iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1).fit(tr[list(z.columns)])
    te["score"] = -iso.score_samples(te[list(z.columns)])      # higher = more abnormal
    return te[["user", "day", "label", "score"]]

insider_frozen_scores(0.5).to_csv("../data/processed/insider_scores.csv", index=False)

In [9]:
import json, os
d = make_data(0.5)
ratio = d[d["label"] == 1][feats].mean() / d[d["label"] == 0][feats].mean()
print(ratio.round(1).sort_values(ascending=False))
top = list(ratio.nlargest(3).index)
json.dump({"InsiderMisuse": top}, open("../data/processed/evidence_insider.json", "w"))

offhours_logons      7.5
usb_events           4.8
files_copied         3.5
distinct_pcs         1.8
email_attachments    1.4
logons               1.1
dtype: float64
